# Goal-Conditioned Analog AI Sizing (5T OTA in 65nm)
This notebook trains a Universal Reinforcement Learning Agent using Stable-Baselines3 (PPO) to size a 5-Transistor OTA for *any* given specification instantly.

## Setup Instructions for Google Colab
1. Upload your entire `Analog_AI_Optimization` folder to your **Google Drive**.
2. Run the cells below to mount your drive and install dependencies.

In [ ]:
!pip install stable-baselines3[extra] gymnasium numpy scipy tensorboard

In [ ]:
from google.colab import drive
import sys
import os

# Mount Google Drive
drive.mount('/content/drive')

# Add your project folder to the Python path
# MODIFY THIS PATH if your folder is named differently or placed in a subfolder!
PROJECT_PATH = '/content/drive/MyDrive/Analog_AI_Optimization'
sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

## 1. Load the Core Engine and LUTs
We load the 65nm TSMC physics (this takes about 20-30 seconds).

In [ ]:
from tech_luts.lut_utils import LUT
from core.device_model import DeviceModel
from circuits.ota5t import OTA5T
from optimizer.rl_environment import OTA5tGymEnv

print("Loading LUTs...")
nch = LUT(os.path.join('tech_luts', 'TSMC_fast_65nm_nch.pkl'))
pch = LUT(os.path.join('tech_luts', 'TSMC_fast_65nm_pch.pkl'))

dm = DeviceModel(nch, pch)
ota = OTA5T(dm, vdd=1.2, cl=1e-12)
ota.Vicm = 0.5
print("Physics Engine Ready!")

## 2. Initialize the Goal-Conditioned RL Environment
The agent will explore 7 parameters. Its state space is 16-D (Current Perf + Target Specs).

In [ ]:
bounds = [
    (60e-9, 1.0e-6),  # L1
    (5.0, 25.0),      # gmid1
    (60e-9, 1.0e-6),  # L3
    (5.0, 25.0),      # gmid3
    (60e-9, 1.0e-6),  # L5
    (5.0, 25.0),      # gmid5
    (10e-6, 500e-6)   # Itail
]

env = OTA5tGymEnv(ota, bounds=bounds, max_steps=200)

## 3. Visualize Progress (TensorBoard)
Run this cell to open the TensorBoard dashboard directly inside Colab. As the AI trains in the next cell, you will see the **Reward (ep_rew_mean)** graphs going up in real time!

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./ppo_ota_tensorboard/

## 4. Train the Universal Agent
We train for 500,000 steps on Colab's high-speed hardware.

In [ ]:
from stable_baselines3 import PPO

# Create PPO Agent with TensorBoard logging enabled
model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.0005, batch_size=256, tensorboard_log="./ppo_ota_tensorboard/")

# Start Heavy Training
print("Starting 500k Steps Training...")
model.learn(total_timesteps=500000)

# Save the final universal model back to Google Drive
model.save("universal_ppo_agent_65nm")
print("Model saved to universal_ppo_agent_65nm.zip on Google Drive!")